In [1]:
import sys
import pandas as pd
import os
import numpy as np
import geopandas as gpd
import seaborn as sns
sns.set_theme()

sys.path.insert(0, 'job_vertex')
import job_logic

import importlib
importlib.reload(job_logic)


/Users/leo/miniconda3/envs/gpl/lib/python3.9/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


<module 'job_logic' from '/Users/leo/Documents/gpl/featurization_competition/featurization-urap-remote-execution/job_vertex/job_logic.py'>

In [ ]:
# Read the featurizer
with open('test_featurizer.py', 'r') as f:
    code = f.read()

job_logic.run_job(
    user_code=code,
    user='selker@berkeley.edu',
    data_dir='/Users/leo/Documents/gpl/featurization_competition/featurization-urap-remote-execution/data_for_bucket',
    full_run=False,
    use_holdout=True,
)re


Index(['txn_type', 'operator', 'caller_msisdn', 'recipient_msisdn',
       'international', 'duration', 'datetime', 'year', 'month', 'day', 'hour',
       'week_of_the_year', 'caller_antenna_id', 'site_id', 'region',
       'prefecture', 'canton'],
      dtype='object')
Email sent to selker@berkeley.edu


{'success': False,
 'error': 'division by zero',
 'traceback': 'Traceback (most recent call last):\n  File "/Users/leo/Documents/gpl/featurization_competition/featurization-urap-remote-execution/job_vertex/job_logic.py", line 899, in evaluate_featurizer\n    user_features = featurizer.featurize(**featurize_kwargs)\n  File "<string>", line 18, in featurize\nZeroDivisionError: division by zero\n',
 'error_type': 'ZeroDivisionError'}

## Manual test

In [ ]:
import test_featurizer
from job_logic import load_data, DropAllNaNColumns, RANDOM_SEED, CV_FOLDS, N_BOOT, LASSO_PARAM_GRID

togo_dfs = load_data('data_for_bucket/input_data/togo')

featurizer = test_featurizer.Featurizer()
user_features = featurizer.featurize(
    cdr=togo_dfs['combined_real_cdr'],
    mobile_money=togo_dfs['combined_real_mobile_money'],
    mobile_data=togo_dfs['combined_real_mobile_data'],
    recharges=None,
    antennas=togo_dfs['combined_real_antennas'],
    shapefiles=togo_dfs['shapefiles'],
)

consumption = togo_dfs['combined_real_consumption']['log_consumption']
cider_features = togo_dfs['cider_features']


In [101]:
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import Lasso
from sklearn.model_selection import GridSearchCV, KFold
from sklearn.metrics import r2_score
from scipy.stats import pearsonr, spearmanr

features=cider_features

merged = features.join(consumption, how='inner')
merged = merged.dropna(subset='log_consumption').reset_index(drop=True)

X = merged[features.columns]
y = merged[consumption.name]

print(f"Samples: {len(X)}, Features: {X.shape[1]}")


Samples: 4000, Features: 1448


In [104]:
lasso_pipeline = Pipeline([
    ('drop_all_nan', DropAllNaNColumns()),
    ("scaler", StandardScaler()),
    ('imputer', SimpleImputer()),
    ("model", Lasso(random_state=RANDOM_SEED, max_iter=5000)),
])

outer_kf = KFold(n_splits=CV_FOLDS, shuffle=True, random_state=RANDOM_SEED)
inner_kf = KFold(n_splits=CV_FOLDS, shuffle=True, random_state=RANDOM_SEED)
rng = np.random.default_rng(RANDOM_SEED)

fold_preds = []
for train_idx, test_idx in outer_kf.split(X):
    gs = GridSearchCV(lasso_pipeline, LASSO_PARAM_GRID, cv=inner_kf, n_jobs=-1, verbose=0)
    gs.fit(X.iloc[train_idx], y.iloc[train_idx])
    fold_preds.append((y.iloc[test_idx], gs.best_estimator_.predict(X.iloc[test_idx])))
    print(f"Best alpha: {gs.best_params_['model__alpha']}")
    print(f"R²: {gs.best_score_:.4f}")

    print(f'OOS R²: ', r2_score(y.iloc[test_idx], gs.best_estimator_.predict(X.iloc[test_idx])))

y_true = np.concatenate([t for t, _ in fold_preds])
y_pred = np.concatenate([p for _, p in fold_preds])
n = len(y_true)

metrics = {}
for metric_name, fn in [
    ("r2", r2_score),
    ("pearson", lambda a, b: pearsonr(a, b)[0]),
    ("spearman", lambda a, b: spearmanr(a, b)[0]),
]:
    vals = []
    for _ in range(N_BOOT):
        idx = rng.integers(0, n, n)
        v = fn(y_true[idx], y_pred[idx])
        if not np.isnan(v):
            vals.append(v)
    lo, hi = np.percentile(vals, [2.5, 97.5]) if vals else (np.nan, np.nan)
    metrics[metric_name] = {"mean": float(np.nanmean(vals)), "ci_low": float(lo), "ci_high": float(hi)}

metrics


Best alpha: 0.1
R²: 0.1281
OOS R²:  0.12065197312701692
Best alpha: 0.1
R²: 0.1196
OOS R²:  0.12657971325463435
Best alpha: 0.1
R²: 0.1183
OOS R²:  0.11191395530968906


{'r2': {'mean': 0.11890188539287083,
  'ci_low': 0.10726138650727376,
  'ci_high': 0.13096002272495635},
 'pearson': {'mean': 0.3902000986403249,
  'ci_low': 0.3615495147588416,
  'ci_high': 0.4171880436274473},
 'spearman': {'mean': 0.35673582478670807,
  'ci_low': 0.32858052982304187,
  'ci_high': 0.3846332960216283}}

In [ ]:
user_features.isna().sum().sort_values(ascending=False).head(20)
